In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

In [5]:
filepath = Path("wdbc_clean.csv")

if not filepath.exists():
    raise FileNotFoundError(
        "Could not find wdbc_clean.csv. "
    )

df = pd.read_csv(filepath)

print("Dataset shape: ", df.shape)
display(df.head())

Dataset shape: (569, 32)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
assert df.isna().sum().sum() == 0, (
    "Missing values were found."
)

assert df["id"].is_unique, (
    "Duplicate patient IDs were found."
)

assert set(df["diagnosis"].unique()) == {"B", "M"}, (
    "Unexpected diagnosis labels were found."
)

print("Final safety checks passed.")

Final safety checks passed.


In [24]:
# Keep IDs for tracking, but do not use them as model features
patient_ids = df["id"].copy()

# Remove ID and diagnosis from the predictor data
X = df.drop(
    columns=["id", "diagnosis"]
).copy()

# Encode diagnosis
y = df["diagnosis"].map({
    "B": 0,
    "M": 1
})

print("Predictor shape:", X.shape)

print("\nDiagnosis counts:")
print(y.value_counts().sort_index())

print("\n0 = Benign")
print("1 = Malignant")

Predictor shape: (569, 30)

Diagnosis counts:
diagnosis
0    357
1    212
Name: count, dtype: int64

0 = Benign
1 = Malignant


In [25]:
non_numeric_columns = (
    X
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

assert not non_numeric_columns, (
    f"Non-numeric predictors found: {non_numeric_columns}"
)

assert X.shape[1] == 30, (
    f"Expected 30 predictors, but found {X.shape[1]}."
)

print("All 30 predictors are numerical.")

All 30 predictors are numerical.


In [26]:
(
    X_development,
    X_test,
    y_development,
    y_test,
    id_development,
    id_test
) = train_test_split(
    X,
    y,
    patient_ids,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Development set:", X_development.shape)
print("Test set:", X_test.shape)

Development set: (455, 30)
Test set: (114, 30)


In [27]:
print("Full dataset proportions:")
print(y.value_counts(normalize=True).sort_index())

print("\nDevelopment set proportions:")
print(
    y_development
    .value_counts(normalize=True)
    .sort_index()
)

print("\nTest set proportions:")
print(
    y_test
    .value_counts(normalize=True)
    .sort_index()
)

Full dataset proportions:
diagnosis
0    0.627417
1    0.372583
Name: proportion, dtype: float64

Development set proportions:
diagnosis
0    0.626374
1    0.373626
Name: proportion, dtype: float64

Test set proportions:
diagnosis
0    0.631579
1    0.368421
Name: proportion, dtype: float64


In [28]:
development_ids = set(id_development)
test_ids = set(id_test)

overlap = development_ids.intersection(test_ids)

assert len(overlap) == 0, (
    "Some patient IDs appear in both sets."
)

assert len(X_development) + len(X_test) == len(df), (
    "The split does not contain every observation."
)

print("No patients overlap between the sets.")
print("All observations were included exactly once.")

No patients overlap between the sets.
All observations were included exactly once.


In [29]:
development_data = (
    X_development
    .reset_index(drop=True)
    .copy()
)

development_data.insert(
    0,
    "id",
    id_development.reset_index(drop=True)
)

development_data["diagnosis"] = (
    y_development.reset_index(drop=True)
)

display(development_data.head())

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst,diagnosis
0,845636,16.02,23.24,102.70,797.8,0.08206,0.06669,0.03299,0.03323,0.1528,...,33.88,123.80,1150.0,0.11810,0.1551,0.1459,0.09975,0.2948,0.08452,1
1,87139402,12.32,12.39,78.85,464.1,0.10280,0.06981,0.03987,0.03700,0.1959,...,15.64,86.97,549.1,0.13850,0.1266,0.1242,0.09391,0.2827,0.06771,0
2,905190,12.85,21.37,82.63,514.5,0.07551,0.08316,0.06126,0.01867,0.1580,...,27.01,91.63,645.8,0.09402,0.1936,0.1838,0.05601,0.2488,0.08151,0
3,907914,14.90,22.53,102.10,685.0,0.09947,0.22250,0.27330,0.09711,0.2041,...,27.57,125.40,832.7,0.14190,0.7090,0.9019,0.24750,0.2866,0.11550,1
4,852781,18.61,20.25,122.10,1094.0,0.09440,0.10660,0.14900,0.07731,0.1697,...,27.26,139.90,1403.0,0.13380,0.2117,0.3446,0.14900,0.2341,0.07421,1


In [30]:
test_data = (
    X_test
    .reset_index(drop=True)
    .copy()
)

test_data.insert(
    0,
    "id",
    id_test.reset_index(drop=True)
)

test_data["diagnosis"] = (
    y_test.reset_index(drop=True)
)

display(test_data.head())

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst,diagnosis
0,865137,11.41,10.82,73.34,403.3,0.09373,0.06685,0.03512,0.02623,0.1667,...,15.97,83.74,510.5,0.1548,0.2390,0.21020,0.08958,0.3016,0.08523,0
1,884948,20.94,23.56,138.90,1364.0,0.10070,0.16060,0.27120,0.13100,0.2205,...,27.00,165.30,2010.0,0.1211,0.3172,0.69910,0.21050,0.3126,0.07849,1
2,901303,16.17,16.07,106.30,788.5,0.09880,0.14380,0.06651,0.05397,0.1990,...,19.14,113.10,861.5,0.1235,0.2550,0.21140,0.12510,0.3153,0.08960,0
3,862548,14.42,19.77,94.48,642.5,0.09752,0.11410,0.09388,0.05839,0.1879,...,30.86,109.50,826.4,0.1431,0.3026,0.31940,0.15650,0.2718,0.09353,1
4,9112085,13.38,30.72,86.34,557.2,0.09245,0.07426,0.02819,0.03264,0.1375,...,41.61,96.69,705.6,0.1172,0.1421,0.07003,0.07763,0.2196,0.07675,0


In [31]:
development_data.to_csv(
    "development_unscaled.csv",
    index=False
)

test_data.to_csv(
    "test_unscaled.csv",
    index=False
)

print("Saved:")
print("- development_unscaled.csv")
print("- test_unscaled.csv")

Saved:
- development_unscaled.csv
- test_unscaled.csv


In [32]:
assert development_data.shape == (455, 32)
assert test_data.shape == (114, 32)

assert development_data.isna().sum().sum() == 0
assert test_data.isna().sum().sum() == 0

assert set(development_data["diagnosis"].unique()) <= {0, 1}
assert set(test_data["diagnosis"].unique()) <= {0, 1}

assert development_data["id"].is_unique
assert test_data["id"].is_unique

print("All preprocessing checks passed.")

All preprocessing checks passed.
